# scRareBench | High-Level scVI | Dataset 2 — mBDRC renal cortex

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **scVI** on scRareBench Dataset 2 (`mbdrc_renal_cortex`) through the generic **MethodSpec → benchmark_method** API.

scVI remains fully user-owned method code. scRareBench loads the benchmark dataset, orchestrates independent method seeds, validates latent/cell alignment, evaluates each run, and builds the finalized multi-seed delivery.

### Dataset 2 / scVI contract

- Dataset selector: `2` (`mbdrc_renal_cortex`)
- Benchmark label: package-registered Dataset 2 label (`cell_type`)
- Integration/evaluation batch: package-registered combined `donor_id × assay`
- HVG policy: **global Seurat-v3** on raw counts; the integration batch is **not** passed to HVG selection
- HVGs: `4000`
- scVI: hidden `128`, latent `30`, layers `2`, dropout `0.10`
- Dispersion: `gene-batch`
- Gene likelihood: `NB`
- Max epochs: `200`
- Train/validation: `90% / 10%`
- Batch size: `256`
- Early stopping patience: `20`
- Method seeds: `42, 123, 2026`
- Benchmark seed: fixed at `42`

The global-HVG policy is intentional: the previous Dataset 2 audit found global Seurat-v3 HVG stable, while batch-aware HVG with the combined `donor_id × assay` batch could become LOESS-singular.

> Use a **GPU Colab runtime**. For a smoke test, temporarily set `METHOD_SEEDS = [42]`.

In [ ]:
# Optional runtime diagnostics. Leave commented unless needed.
# import sys
# from importlib import metadata
# print(sys.version)
# for pkg in ("numpy", "torch", "jax", "scanpy", "scvi-tools"):
#     try:
#         print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError:
#         print(pkg, "not installed")


## 1. Install scRareBench and scVI


In [ ]:
import subprocess
import sys

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)
INSTALL_METHOD_DEPENDENCIES = True

from scrarebench.runtime import setup_runtime

setup_runtime(
    extra_requirements=(
        METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else ()
    ),
    extra_imports=("scvi",) if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)


## 2. Load Dataset 2


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(2)
info = dataset_info(adata)

print(adata)
print(info)

if info.get("label_key") is None:
    raise RuntimeError(f"Dataset 2 label metadata is missing: {info}")
if info.get("batch_key") is None:
    raise RuntimeError(f"Dataset 2 batch metadata is missing: {info}")
if info.get("scib_hvg_batch_mode") != "global":
    raise RuntimeError(
        "Dataset 2 is expected to use global HVG selection; "
        f"observed metadata: {info}"
    )

print("Benchmark label:", info["label_key"])
print("Integration/evaluation batch:", info["batch_key"])
print("HVG batch mode:", info["scib_hvg_batch_mode"])


## 3. Recover and validate raw counts


In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

def sampled_count_diagnostics(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[
            rng.choice(len(values), size=max_values, replace=False)
        ]

    values = np.asarray(values, dtype=float)
    finite_mask = np.isfinite(values)
    finite = values[finite_mask]

    return {
        "finite_fraction": (
            float(np.mean(finite_mask)) if len(values) else 1.0
        ),
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "nonnegative": bool(
            len(finite) == 0 or np.min(finite) >= 0
        ),
        "integer_like": bool(
            len(finite) == 0
            or np.allclose(finite, np.rint(finite), atol=1e-6)
        ),
    }

def copy_matrix(matrix):
    if sp.issparse(matrix):
        return matrix.copy().tocsr()
    return np.asarray(matrix).copy()

def ensure_counts_layer(adata, layer_key="counts"):
    if layer_key in adata.layers:
        diag = sampled_count_diagnostics(adata.layers[layer_key])
        if diag["nonnegative"] and diag["integer_like"]:
            return f"layers/{layer_key}", diag
        raise ValueError(
            f"Existing {layer_key!r} layer is not valid raw counts: {diag}"
        )

    if adata.raw is not None:
        raw_names = pd.Index(adata.raw.var_names.astype(str))
        current_names = pd.Index(adata.var_names.astype(str))
        positions = raw_names.get_indexer(current_names)

        if np.all(positions >= 0):
            matrix = adata.raw.X[:, positions]
            diag = sampled_count_diagnostics(matrix)
            if diag["nonnegative"] and diag["integer_like"]:
                adata.layers[layer_key] = copy_matrix(matrix)
                return "raw.X aligned to adata.var_names", diag

    diag = sampled_count_diagnostics(adata.X)
    if diag["nonnegative"] and diag["integer_like"]:
        adata.layers[layer_key] = copy_matrix(adata.X)
        return "X", diag

    raise ValueError(
        "Could not locate nonnegative integer-like raw counts for Dataset 2. "
        f"layers={list(adata.layers)}, "
        f"raw_present={adata.raw is not None}, "
        f"X_diagnostics={diag}"
    )

COUNT_LAYER = info.get("count_layer") or "counts"
count_source, count_diagnostics = ensure_counts_layer(
    adata, COUNT_LAYER
)

print("Raw-count layer:", COUNT_LAYER)
print("Raw-count source:", count_source)
print("Count diagnostics:", count_diagnostics)

if count_diagnostics["finite_fraction"] != 1.0:
    raise ValueError("Raw counts contain NaN/Inf.")
if not count_diagnostics["nonnegative"]:
    raise ValueError("Raw counts contain negative values.")
if not count_diagnostics["integer_like"]:
    raise ValueError("Raw counts are not integer-like.")


## 4. Reproducibility and scVI configuration


In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_policy": "global_seurat_v3_raw_counts",
    "hvg_batch_key": None,
    "hvg_span": 0.3,
    "n_hidden": 128,
    "n_latent": 30,
    "n_layers": 2,
    "dropout_rate": 0.10,
    "dispersion": "gene-batch",
    "gene_likelihood": "nb",
    "max_epochs": 200,
    "train_size": 0.90,
    "batch_size": 256,
    "early_stopping": True,
    "early_stopping_patience": 20,
}

OUTPUT_DIR = "/content/scrarebench_scvi_dataset2_highlevel"


## 5. User-owned scVI runner

The benchmark label is never supplied to scVI.

Dataset 2 uses **global** Seurat-v3 HVGs from raw counts. The registered Dataset 2 batch is used by scVI for integration, but deliberately not by the HVG step.


In [ ]:
import random
import numpy as np
import scanpy as sc
import scvi
import torch

from scrarebench import MethodOutput

def run_scvi(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"

    if count_layer not in method_adata.layers:
        raise KeyError(
            f"Raw-count layer {count_layer!r} is missing from the method copy."
        )

    original_barcodes = (
        method_adata.obs_names.astype(str).to_numpy(copy=True)
    )
    if len(np.unique(original_barcodes)) != len(original_barcodes):
        raise ValueError("Dataset 2 has duplicate cell barcodes.")

    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    scvi.settings.seed = seed

    # Dataset 2 fixed policy: GLOBAL HVGs from raw counts.
    hvg_kwargs = {
        "layer": count_layer,
        "flavor": "seurat_v3",
        "n_top_genes": min(
            int(config["n_hvg"]), method_adata.n_vars
        ),
        "span": float(config["hvg_span"]),
        "subset": False,
        "check_values": True,
    }

    if config.get("hvg_batch_key") is not None:
        hvg_kwargs["batch_key"] = config["hvg_batch_key"]

    sc.pp.highly_variable_genes(method_adata, **hvg_kwargs)

    hvg_mask = (
        method_adata.var["highly_variable"]
        .fillna(False)
        .to_numpy(dtype=bool)
    )
    expected_hvg = min(
        int(config["n_hvg"]), method_adata.n_vars
    )
    observed_hvg = int(hvg_mask.sum())

    if observed_hvg != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} global HVGs but selected "
            f"{observed_hvg}."
        )

    method_adata = method_adata[:, hvg_mask].copy()

    current_barcodes = (
        method_adata.obs_names.astype(str).to_numpy()
    )
    if not np.array_equal(
        current_barcodes, original_barcodes
    ):
        raise RuntimeError(
            "Cell order changed during scVI feature preprocessing."
        )

    scvi.model.SCVI.setup_anndata(
        method_adata,
        layer=count_layer,
        batch_key=batch_key,
    )

    model = scvi.model.SCVI(
        method_adata,
        n_hidden=int(config["n_hidden"]),
        n_latent=int(config["n_latent"]),
        n_layers=int(config["n_layers"]),
        dropout_rate=float(config["dropout_rate"]),
        dispersion=config["dispersion"],
        gene_likelihood=config["gene_likelihood"],
        latent_distribution="normal",
    )

    train_size = float(config["train_size"])
    model.train(
        max_epochs=int(config["max_epochs"]),
        train_size=train_size,
        validation_size=1.0 - train_size,
        batch_size=int(config["batch_size"]),
        early_stopping=bool(config["early_stopping"]),
        early_stopping_patience=int(
            config["early_stopping_patience"]
        ),
        check_val_every_n_epoch=1,
        accelerator="auto",
        devices="auto",
        enable_progress_bar=True,
    )

    latent = np.asarray(
        model.get_latent_representation(),
        dtype=np.float32,
    )

    expected_shape = (
        method_adata.n_obs,
        int(config["n_latent"]),
    )
    if latent.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected latent shape {latent.shape}; "
            f"expected {expected_shape}."
        )
    if not np.isfinite(latent).all():
        raise ValueError("scVI latent contains NaN/Inf.")

    return MethodOutput(
        latent=latent,
        barcodes=current_barcodes,
        representation_key="X_scvi",
    )


## 6. Run the High-Level multi-seed benchmark

For an initial smoke test, temporarily use `METHOD_SEEDS = [42]`. Restore all three seeds for the final benchmark.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scVI",
    runner=run_scvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    install_dependencies=False,
    finalize=True,
)


## 7. Inspect the finalized result


In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)


## 8. Download the complete delivery archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)

if not archive_path.exists():
    raise FileNotFoundError(
        f"Output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(
    f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB"
)
files.download(str(archive_path))
